# Análise de cada estação (sta)

Este notebook carrega as cinco medições `1calibrar_cn.csv` a `5calibrar_cn.csv`, calcula a média e o desvio padrão do `training_time` para cada frequência de cada estação, e ajusta uma curva por mínimos quadrados entre o tempo médio e `1/freq`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-whitegrid')

INSTANCIA = {
    "N": 20,
    "alpha": [2e-18] * 20,
    "c": [-1] * 24,
    "num_samples": [17260.0, 15810.0, 26070.0, 24200.0, 10080.0, 26230.0, 23000.0, 23720.0, 8140.0, 25470.0, 17290.0, 17650.0, 17050.0, 18750.0, 16420.0, 18170.0, 8780.0, 24870.0, 7830.0, 17590.0],
    "f_min": [1.4, 1.5, 1.5, 1.4, 1.4, 1.5, 1.4, 1.6, 2.6, 2.6, 2.6, 2.6, 2.6, 2.6, 2.6, 2.6, 2.6, 2.6, 2.6, 2.6],
    "f_max": [2.0, 1.8, 3.6, 2.3, 3.7, 2.4, 2.7, 2.0, 2.9, 3.6, 3.9, 3.8, 3.6, 3.4, 3.5, 3.1, 3.4, 3.5, 3.1, 3.7],
    "epsilon_0": 0.999,
    "theta_prev": [0.1] * 20,
}


def prepara_df(df: pd.DataFrame, instancia: dict) -> pd.DataFrame:
    num_sta = instancia["N"]
    num_rows = len(df)
    for i in range(num_sta):
        f_min_sta = instancia["f_min"][i]
        f_max_sta = instancia["f_max"][i]
        delta_freq = (f_max_sta - f_min_sta) / (num_rows - 1) if num_rows > 1 else 0.0
        df[f'sta{i}_freq'] = [f_min_sta + k * delta_freq for k in range(num_rows)]
    return df


dataframes = []
for i in range(1, 6):
    path = f'{i}calibrar_cn.csv'
    df = pd.read_csv(path, index_col=0)
    df = prepara_df(df, INSTANCIA)
    df['measure_id'] = i
    dataframes.append(df)

all_df = pd.concat(dataframes, ignore_index=True, sort=False)


def build_long(df: pd.DataFrame, instancia: dict) -> pd.DataFrame:
    rows = []
    for i in range(instancia['N']):
        station = f'sta{i}'
        freq_col = f'{station}_freq'
        datasz_col = f'{station}_datasz'
        time_col = f'training_time_{station}'
        if freq_col not in df.columns or datasz_col not in df.columns or time_col not in df.columns:
            continue
        tmp = df[[freq_col, datasz_col, time_col, 'measure_id']].copy()
        tmp.columns = ['freq', 'datasz', 'training_time', 'measure_id']
        tmp['station'] = station
        rows.append(tmp)
    return pd.concat(rows, ignore_index=True)

long_df = build_long(all_df, INSTANCIA)
long_df = long_df.dropna(subset=['freq', 'datasz', 'training_time'])
long_df['inv_freq'] = 1.0 / long_df['freq']
long_df['datasz_div_freq'] = long_df['datasz'] / long_df['freq']

stats = (
    long_df
    .groupby(['station', 'freq', 'datasz'], as_index=False)
    .training_time
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .rename(columns={'mean': 'mean_time', 'std': 'std_time', 'count': 'n_measurements'})
)


def fit_linear(x: np.ndarray, y: np.ndarray):
    if len(x) < 2:
        return np.nan, np.nan
    a, b = np.polyfit(x, y, 1)
    return a, b


def fit_through_origin(x: np.ndarray, y: np.ndarray):
    denom = np.sum(x * x)
    return np.sum(x * y) / denom if denom != 0 else np.nan

fit_rows = []
for station, group in stats.groupby('station'):
    x_inv = group['inv_freq'].values
    y = group['mean_time'].values
    a_inv, b_inv = fit_linear(x_inv, y)
    x_model = group['datasz_div_freq'].values
    a_model = fit_through_origin(x_model, y)
    fit_rows.append({
        'station': station,
        'a_inv': a_inv,
        'b_inv': b_inv,
        'a_model': a_model,
        'n_points': len(group),
    })

fits = pd.DataFrame(fit_rows)
fits['station_number'] = fits['station'].str.extract(r'sta(\d+)').astype(int)

print('Resumo de ajustes para cada estação:')
print(fits.sort_values('station_number').head(10).to_string(index=False))

: 

In [ ]:
stations = sorted(stats['station'].unique(), key=lambda s: int(s.replace('sta', '')))
fig, axes = plt.subplots(4, 5, figsize=(24, 18), sharey=False)
axes = axes.flatten()
for ax, station in zip(axes, stations):
    station_stats = stats[stats['station'] == station].sort_values('freq')
    if station_stats.empty:
        ax.axis('off')
        continue

    ax.errorbar(
        station_stats['freq'],
        station_stats['mean_time'],
        yerr=station_stats['std_time'],
        fmt='o',
        capsize=3,
        label='mean ± std',
        color='C0',
    )

    fit = fits[fits['station'] == station].iloc[0]
    x_plot = np.linspace(station_stats['freq'].min(), station_stats['freq'].max(), 120)
    y_fit_inv = fit['a_inv'] * (1.0 / x_plot) + fit['b_inv']
    ax.plot(x_plot, y_fit_inv, '-', color='C1', label=f'fit t=a*(1/f)+b')

    ax.set_title(station)
    ax.set_xlabel('freq (Hz)')
    ax.set_ylabel('training_time (s)')
    ax.legend(fontsize='small')
    ax.grid(True, linestyle=':', linewidth=0.5)

for ax in axes[len(stations):]:
    ax.axis('off')

fig.suptitle('Treinamento médio por estação com erro padrão e ajuste linear em 1/freq', fontsize=18)
fig.subplots_adjust(top=0.94, hspace=0.35, wspace=0.25)
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 6))
combined = stats.copy()
combined['datasz_div_freq'] = combined['datasz'] / combined['freq']
ax2.scatter(combined['datasz_div_freq'], combined['mean_time'], alpha=0.8)
coef = fit_through_origin(combined['datasz_div_freq'].values, combined['mean_time'].values)
xs = np.linspace(combined['datasz_div_freq'].min(), combined['datasz_div_freq'].max(), 120)
ax2.plot(xs, coef * xs, color='red', label=f't = {coef:.3e} * datasz/freq (sem intercept)')
ax2.set_xlabel('datasz / freq')
ax2.set_ylabel('mean training time (s)')
ax2.set_title('Ajuste teórico t ∝ datasz / freq para todos os dados')
ax2.legend()
ax2.grid(True, linestyle=':')
plt.tight_layout()
plt.show()